### 02. Label AI passages with an LLM judge

**Goal:** pick 30 random rules, cut out the passages that mention AI, and have Gemini label each rule using a codebook. Team members label the same 30 rules by hand to check Gemini's labels.

**Opens:** `fr_rules.csv` (from notebook 01)

**Saves:** `hand_labels.csv` (passages + hand labels), `gemini_labels.csv`

**AI assistance:** The code in this notebook was drafted alongside Claude and was reviewed, run, and checked by our team.

Links to official documentation are in the last cell.

#### Step 1: Import libraries

In [ ]:
import os        # checks if a file is already saved
import re        # finds words in text, like "artificial intelligence"
import json      # reads Gemini's replies
import time      # pauses the code between Gemini requests

import pandas as pd               
import requests                   
from bs4 import BeautifulSoup    # removes web-page code (HTML) from downloaded text

#### Step 2: Pick 30 random rules

`random_state=211` makes the sample the same every time the code runs, so every team member gets the same 30 rules

In [ ]:
# Open the table of rules saved by notebook 01
rules = pd.read_csv("fr_rules.csv")

# The three agencies to sample from
chosen_agencies = [
    "Health and Human Services Department",
    "Commerce Department",
    "Homeland Security Department",
]

# Keep only rules whose agency is in the list
three_agencies = rules[rules["agency"].isin(chosen_agencies)]
print("Rules from the three agencies:", len(three_agencies))

# Pick 30 of them at random, and number the rows 0 to 29
sample = three_agencies.sample(n=30, random_state=211)
sample = sample.reset_index(drop=True)

# Check how many came from each agency and each administration
print(sample["agency"].value_counts())
print(sample["administration"].value_counts())

Rules from the three agencies: 109
agency
Health and Human Services Department    15
Commerce Department                      9
Homeland Security Department             6
Name: count, dtype: int64
administration
Trump II    15
Biden       12
Trump I      3
Name: count, dtype: int64


#### Step 3: Cut out the parts of each rule that mention AI

Rules are long, often thousands of words, and usually mention AI only a few times. This step downloads each rule's full text and keeps only the text around each mention of a search term, about 500 characters.

* Step 3a downloads one rule
* Step 3b finds the passages in that rule
* Step 3c does the same for all 20 rules

##### Step 3a: Download one rule's full text

In [3]:
# A function that downloads one rule's full text and cleans it up
def download_text(url):
    """Download a rule's full text and return it as plain text."""
    response = requests.get(url, timeout=60)      # send the request and wait up to 60 seconds
    response.raise_for_status()                   # stop with an error if the request failed
    text = BeautifulSoup(response.text, "html.parser").get_text()   # remove web-page code (HTML)
    text = " ".join(text.split())            # replace line breaks and extra spaces with single spaces
    return text

# Try it on the first rule in the sample
first_text = download_text(sample.loc[0, "raw_text_url"])

print("Number of characters:", len(first_text))
print(first_text[:500])                  # show the first 500 characters

Number of characters: 444542
Federal Register, Volume 91 Issue 197 (Wednesday, October 14, 2026) [Federal Register Volume 91, Number 197 (Wednesday, October 14, 2026)] [Rules and Regulations] [Pages 65306-65368] From the Federal Register Online via the Government Publishing Office [www.gpo.gov] [FR Doc No: 2026-20955] [[Page 65305]] Vol. 91 Wednesday, No. 197 October 14, 2026 Part II Department of Homeland Security ----------------------------------------------------------------------- 8 CFR Part 274a Removal of Automatic E


##### Step 3b: Find the passages that mention AI

The search terms are the same as in notebook 01. The `|` between them means "or."

For each mention, the code keeps 500 characters before and after it. It keeps at most 3 passages per rule, and skips a mention if it's already inside the previous passage.

In [ ]:
# The search terms, joined with | ("or")
search_pattern = "artificial intelligence|machine learning|automated decision|deep learning|large language model"

# A function that finds up to 3 passages around the search terms
def get_passages(text):
    """Return up to 3 passages around the search terms, joined into one piece of text."""
    passages = []        # an empty list for the passages
    last_end = 0         # where the previous passage ended

    # Go through each place a search term appears (ignoring capital letters)
    for match in re.finditer(search_pattern, text, flags=re.IGNORECASE):
        if match.start() < last_end:              # this mention is already inside the previous passage
            continue                              # skip it and go to the next mention

        start = max(match.start() - 500, 0)       # 500 characters before (but not before the start of the text)
        end = match.end() + 500                   # 500 characters after
        passages.append(text[start:end])          # cut out the passage and add it to the list
        last_end = end

        if len(passages) == 3:               # stop after 3 passages
            break

    return "\n\n...\n\n".join(passages)      # join the passages with "..." between them

# Try it on the first rule
print(get_passages(first_text))

plies a range of 25% to account for variation.\183\ The forecasted monthly time series point value estimate aggregated over 18 months and ranges are shown in Table 3, along with other relevant population components. To account for the non-C8 and C9 populations affected by the rule, DHS added a 22% factor to the broad population. --------------------------------------------------------------------------- \183\ DHS utilized the SAS JMP PRO 17 software. The ``State Space Smoothing'' suite utilizes machine learning algorithms to offer the best model information criterion to assess fit and forecasting technique for queued time series data, given dozens of potential model specifications. Based on the Akaike Information Criterion (AIC) DHS forecasted the data via a multiplicative error- multiplicative (undamped) trend, non-seasonal component (MMN) setup. The MMN model fits five parameters, but these are not reported here since the goal is limited to forecasting and not hypothesis testing. The

##### Step 3c: Do the same for all 30 rules

In [ ]:
# An empty list for each rule's passages
all_passages = []

# Repeat for each rule in the sample
for url in sample["raw_text_url"]:
    text = download_text(url)                  # download the full text
    all_passages.append(get_passages(text))    # find the passages and add them to the list

# Add the passages to the sample table as a new column
sample["passages"] = all_passages

# Check: rules where no passage was found (should be 0)
no_passages = sample[sample["passages"] == ""]
print("Rules with no passages found:", len(no_passages))

Rules with no passages found: 1


#### Step 4: Save the passages with empty label columns

This step saves one file, `hand_labels.csv`, with the passages for each rule and empty columns: `label_coder1`, `label_coder2`, and `label_coder3`. Team members fill in their own column.

In [6]:
# Keep only the columns needed for labeling
hand_labels = sample[["document_number", "title", "agency", "matched_terms", "html_url", "passages"]]

# Add three empty columns, one for each coder
hand_labels["label_coder1"] = ""
hand_labels["label_coder2"] = ""
hand_labels["label_coder3"] = ""

# Save the file, but only if it doesn't exist yet
if os.path.exists("hand_labels.csv"):
    print("hand_labels.csv already exists, so it was not replaced")
else:
    hand_labels.to_csv("hand_labels.csv", index=False)
    print("Saved hand_labels.csv with", len(hand_labels), "rules")

hand_labels.csv already exists, so it was not replaced


/var/folders/98/tq_hd9yj52z0x2tnxd_hxcq80000gn/T/ipykernel_463/2534319115.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hand_labels["label_coder1"] = ""
/var/folders/98/tq_hd9yj52z0x2tnxd_hxcq80000gn/T/ipykernel_463/2534319115.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hand_labels["label_coder2"] = ""
/var/folders/98/tq_hd9yj52z0x2tnxd_hxcq80000gn/T/ipykernel_463/2534319115.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_ind

#### Step 5: Connect to Gemini

In [7]:
from dotenv import load_dotenv     # reads the .env file
from google import genai           # connects to Gemini
from google.genai import types     # settings for Gemini

# Read the key from the .env file
load_dotenv()

# The Gemini model to use (the same one as in the Week 7 slides)
gemini_model = "gemini-3.5-flash-lite"

# If Gemini is busy (error 429 or 503), wait and try again, up to 3 times
retry = types.HttpRetryOptions(
    attempts=3,                    # try up to 3 times
    initial_delay=1.0,             # wait 1 second before the first retry
    exp_base=2,                    # double the wait each time (1 second, then 2, then 4)
    http_status_codes=[429, 503],  # the errors that mean "busy, try again"
)

# Connect to Gemini
client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"],
    http_options=types.HttpOptions(retry_options=retry),
)

print("Connected to Gemini")

Connected to Gemini


#### Step 6: The codebook

The codebook explains each label. Gemini and the hand coders use the same codebook.

| Label | Meaning |
|---|---|
| `regulate` | The rule adds limits, oversight, or protections for AI (for example: addressing risks, harms, bias, safety, or privacy, or setting rules for how AI may be used). |
| `promote` | The rule encourages more use of AI (for example: funding AI, or pointing to its benefits, efficiency, or innovation). |
| `deregulate` | The rule removes, delays, or declines to adopt limits or oversight on AI (for example: dropping a proposed AI requirement, or removing an earlier AI rule). |
| `mixed` | The rule clearly does more than one of the above. |
| `none` | The rule mentions AI without doing any of the above (for example: listing AI among other technologies). |

**Rules for tricky cases:**
* Requiring that AI be tested, disclosed, or overseen counts as `regulate`, even if the rule allows AI.
* If a public commenter asks for something, label the agency's response, not the commenter's request.

If a label's meaning changes, update both this table and the `codebook` text in the code.

In [ ]:
# The instructions sent to Gemini with every rule (the same codebook as the table above)
codebook = """You are labeling passages from U.S. Federal Register rules for a research project.
Each passage mentions artificial intelligence (AI) or a related term.

Decide how the agency treats AI. Give one label:

regulate: The rule adds limits, oversight, or protections for AI (for example: addressing risks, harms, bias, safety, or privacy, or setting rules for how AI may be used).
promote: The rule encourages more use of AI (for example: funding AI, or pointing to its benefits, efficiency, or innovation).
deregulate: The rule removes, delays, or declines to adopt limits or oversight on AI (for example: dropping a proposed AI requirement, or removing an earlier AI rule).
mixed: The rule clearly does more than one of the above.
none: The rule mentions AI without doing any of the above (for example: listing AI among other technologies).

Rules for tricky cases:
- Requiring that AI be tested, disclosed, or overseen counts as regulate, even if the rule allows AI.
- If a public commenter asks for something, label the agency's response, not the commenter's request.

Judge only from the text given. Also give a two to three sentence reason.
"""

# The format Gemini's answer must follow: one of the 5 labels, plus a reason
answer_format = {
    "type": "object",
    "properties": {
        "label": {"type": "string", "enum": ["regulate", "promote", "deregulate", "mixed", "none"]},
        "reason": {"type": "string"},
    },
    "required": ["label", "reason"],
}

# Gemini's settings: the codebook, the answer format, and temperature 0 for consistent answers
settings = types.GenerateContentConfig(
    system_instruction=codebook,
    response_mime_type="application/json",    # answer in JSON format
    response_schema=answer_format,
    temperature=0,                            # 0 = give the same answer each time
)

#### Step 7: Test Gemini on one rule

Gemini gets each rule's title and passages, but not the date or administration, so it can't guess the politics from the date.

In [9]:
# A function that sends one rule to Gemini and returns its label and reason
def ask_gemini(title, passages):
    """Send one rule's title and passages to Gemini. Return (label, reason)."""
    text = "Title: " + title + "\n\nPassages:\n" + passages # the text Gemini reads: the title, then the passages

    # Send the text with the settings from Step 6 (codebook, answer format, temperature)
    response = client.models.generate_content(
        model=gemini_model,
        contents=text,
        config=settings,
    )

    answer = json.loads(response.text) # Turn Gemini's JSON answer into a Python dictionary
    return answer["label"], answer["reason"]

label, reason = ask_gemini(sample.loc[0, "title"], sample.loc[0, "passages"]) # try label on the first rule in the sample

print("Document number:", sample.loc[0, "document_number"])
print("Title: ", sample.loc[0, "title"])
print("Label: ", label)
print("Reason:", reason)

Document number: 2026-20955
Title:  Removal of Automatic Extensions for Certain Employment Authorization Documents
Label:  none
Reason: The passage simply mentions the use of machine learning algorithms within a software suite to forecast data points for population estimates. It does not regulate, promote, deregulate, or take any normative stance on artificial intelligence.


#### Step 8: Label all 30 rules with Gemini

The function from Step 7 runs once for each rule. The labels and reasons are saved as `gemini_labels.csv`.

This makes 30 requests to Gemini, so it takes about 4 minutes, because the code waits 8 seconds between rules (the free tier allows 15 requests per minute). If Gemini can't answer for a rule, that rule gets the label "error" so it can be checked by hand.

In [ ]:
# An empty list for Gemini's answers
results = []

# Repeat for each of the 30 rules
for i in range(len(sample)):                      
    document_number = sample.loc[i, "document_number"]
    title = sample.loc[i, "title"]
    passages = sample.loc[i, "passages"]

    # Ask Gemini. If something goes wrong, record "error" instead of stopping.
    try:
        label, reason = ask_gemini(title, passages)
    except Exception as problem:
        label = "error"
        reason = str(problem)                     # save the error message as the reason

    # Add this rule's answer to the list
    results.append({
        "document_number": document_number,
        "gemini_label": label,
        "gemini_reason": reason,
    })

    print(i + 1, "of", len(sample), ":", label)   # show progress
    time.sleep(8)                                 # wait 8 seconds before the next request

# Turn the list into a table and save it
gemini_labels = pd.DataFrame(results)
gemini_labels.to_csv("gemini_labels.csv", index=False)
print("Saved gemini_labels.csv")

# Count how many rules got each label
gemini_labels["gemini_label"].value_counts()

1 of 30 : none
2 of 30 : promote
3 of 30 : mixed
4 of 30 : mixed
5 of 30 : promote
6 of 30 : none
7 of 30 : regulate
8 of 30 : regulate
9 of 30 : promote
10 of 30 : none
11 of 30 : none
12 of 30 : none
13 of 30 : none
14 of 30 : none
15 of 30 : none
16 of 30 : none
17 of 30 : regulate
18 of 30 : none
19 of 30 : promote
20 of 30 : regulate
21 of 30 : regulate
22 of 30 : regulate
23 of 30 : promote
24 of 30 : regulate
25 of 30 : none
26 of 30 : none
27 of 30 : none
28 of 30 : regulate
29 of 30 : regulate
30 of 30 : none
Saved gemini_labels.csv


gemini_label
none        14
regulate     9
promote      5
mixed        2
Name: count, dtype: int64

#### Documentation

* **Gemini API:** [Python library (google-genai)](https://googleapis.github.io/python-genai/) · [structured output (answer format)](https://ai.google.dev/gemini-api/docs/structured-output) · [system instructions (codebook)](https://ai.google.dev/gemini-api/docs/text-generation#system-instructions)
* **python-dotenv:** [reading the .env file](https://pypi.org/project/python-dotenv/)
* **requests:** [quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/)
* **BeautifulSoup:** [documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
* **pandas:** [read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [sample](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html) · [reset_index](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html) · [copy](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html) · [to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html)
* **Python:** [re.finditer (finding words in text)](https://docs.python.org/3/library/re.html#re.finditer) · [os.path.exists](https://docs.python.org/3/library/os.path.html#os.path.exists) · [json.loads](https://docs.python.org/3/library/json.html#json.loads) · [try / except](https://docs.python.org/3/tutorial/errors.html#handling-exceptions)